# DiffuLLaMA-7B: the five remaining experiments

Runs the POS probe, logit lens, attention entropy, and the German + Japanese locked transfers.
Head search and the EWT time curve are already done.

**Runtime: A100.** Set it under *Runtime -> Change runtime type* before running anything.

Run top to bottom. Cell 2 restarts the runtime once, on purpose — resume from cell 3 when it does.
Finished runs are skipped, so reconnecting and re-running the whole notebook is safe.


## 1. GPU check


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

mib = int(gpu.rsplit(",", 1)[-1].strip().split()[0]) if gpu else 0
if mib < 30000:
    raise SystemExit(
        f"{gpu!r} has {mib} MiB. DiffuLLaMA-7B is bf16 with eager attention and needs "
        "~14GB of weights plus the attention tensors. Switch to an A100."
    )
print("OK:", gpu)


## 2. Clone and install

Private repo. Add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar), or accept the hidden prompt.

**This cell restarts the runtime.** Colab preloads a newer `transformers` and the pin below
only takes effect after a restart. Re-run from cell 3, not cell 2.


In [ ]:
REPO = "github.com/Dabsoysauce/latentrelationsondlm.git"
BRANCH = "diffullama-remaining-experiments"
ROOT = "/content/dlmresearch"
SRC = f"{ROOT}/src"

import os


def github_token():
    try:
        from google.colab import userdata

        value = userdata.get("GITHUB_TOKEN")
        if value:
            return value
    except Exception as exc:
        print(f"no Colab secret ({type(exc).__name__})")
    import getpass

    return getpass.getpass("GitHub token (input hidden): ")


if not os.path.exists(ROOT):
    token = github_token()
    # Held in this variable only: never echoed, never written to disk.
    !git clone --quiet --branch {BRANCH} https://{token}@{REPO} {ROOT}
    del token

%cd {ROOT}
!git log --oneline -1
!pip install -q -e .
!pip install -q -r requirements/diffullama.txt

# An editable install registers itself through a .pth file, and .pth files are only
# read at interpreter startup -- so `import dlmrel` fails in the session that
# installed it. Add the source root directly.
import importlib, sys

if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import transformers

print("transformers:", transformers.__version__)
if transformers.__version__ != "4.44.2":
    print("Restarting so the pin takes effect. Re-run from cell 3.")
    os.kill(os.getpid(), 9)


## 3. Verify the environment

`transformers` must be exactly 4.44.2. DiffuLLaMA's `attention_patch.py` replaces
`LlamaModel.forward` with the 4.44-era implementation and patches `LlamaFlashAttention2`,
which later releases deleted.


In [ ]:
ROOT = "/content/dlmresearch"
SRC = f"{ROOT}/src"
REPO = "github.com/Dabsoysauce/latentrelationsondlm.git"
%cd {ROOT}

import importlib, sys

if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import dlmrel, torch, transformers

print("dlmrel      ", dlmrel.__file__)
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)

assert transformers.__version__ == "4.44.2", (
    f"transformers is {transformers.__version__}; re-run cell 2 and let it restart."
)

from transformers.models.llama import modeling_llama

assert hasattr(modeling_llama, "LlamaFlashAttention2"), (
    "LlamaFlashAttention2 is missing, so attention_patch.py will fail on import."
)
assert torch.cuda.is_available(), "no GPU visible to torch"
print("\nenvironment OK")


## 4. The EWT selection lock

The two transfer runs replay the head that EWT head search froze. That lock lives in PR #18,
so if it has not been merged yet this cell pulls it straight from the pull request.


In [ ]:
import json
from pathlib import Path


def github_token():
    try:
        from google.colab import userdata

        value = userdata.get("GITHUB_TOKEN")
        if value:
            return value
    except Exception as exc:
        print(f"no Colab secret ({type(exc).__name__})")
    import getpass

    return getpass.getpass("GitHub token (input hidden): ")


LOCK = Path("docs/results/diffullama_confirmatory/selection_lock.json")

if not LOCK.exists():
    print("lock not on this branch; fetching it from PR #18")
    token = github_token()
    !git fetch --quiet https://{token}@{REPO} pull/18/head:pr18
    del token
    LOCK.parent.mkdir(parents=True, exist_ok=True)
    !git show pr18:docs/results/diffullama_confirmatory/selection_lock.json > {LOCK}

lock = json.loads(LOCK.read_text())
assert lock["relation"] == "object_to_verb", lock["relation"]
assert lock["model_id"] == "diffullama_7b", lock["model_id"]

print("locked head:      layer {}, head {}".format(lock["layer"], lock["head"]))
print("frozen progress:  {}".format(lock["frozen_settings"]["selection_progress"]))
print("fixed-offset null:", lock["frozen_settings"]["fixed_offset"])


## 5. Prepare the treebanks

Downloads UD 2.15 at pinned revisions and writes official-boundary manifests. CPU only.

The EWT manifest hashes have to match the ones recorded in the lock. If they do not, the
transfer runs would be replaying a head that was selected on different data.


In [ ]:
import glob
import subprocess

for dataset in ("ewt", "de_gsd", "ja_gsd"):
    print(f"\n=== prepare {dataset} ===", flush=True)
    subprocess.run(
        ["dlmrel", "prepare", "--dataset", f"configs/datasets/{dataset}.yaml"], check=True
    )

audit = json.loads(Path(glob.glob("data/manifests/ewt/*/audit.json")[0]).read_text())
hashes = audit["manifest_hashes"]
assert hashes["select"] == lock["select_manifest_hash"], "EWT select manifest differs from the lock"
assert hashes["dev"] == lock["dev_manifest_hash"], "EWT dev manifest differs from the lock"
print("\nEWT manifests match the selection lock")


## 6. Smoke-test the adapter

The checkpoint is stored under a `denoise_model.*` wrapper namespace. Loaded the obvious way
it leaves the backbone randomly initialised with only a warning — which is exactly how an
earlier 7B search came back at chance. Do not skip this.


In [ ]:
!dlmrel smoke-test --model configs/models/diffullama_7b.yaml --output smoke_diffullama.json

report = json.loads(Path("smoke_diffullama.json").read_text())
print(json.dumps(report, indent=2)[:2000])


## 7. Run the five experiments

Cost order, cheapest first, so a failure surfaces before hours are spent.

| # | experiment | dataset | forward passes |
| --- | --- | --- | ---: |
| 1 | POS probe | ewt | 18,000 |
| 2 | logit lens | ewt | 15,000 |
| 3 | attention entropy | ewt | 27,000 |
| 4 | locked transfer | de_gsd | 2,766 |
| 5 | locked transfer | ja_gsd | 1,611 |

A run whose `summary.json` exists is skipped, and a partial run is continued with `--resume`
against checkpoints written every 300 sentences. So re-running this cell after a disconnect
picks the schedule back up instead of repeating it.


In [ ]:
import time

MODEL = "configs/models/diffullama_7b.yaml"

# label, dataset, experiment yaml, track, experiment id, run id, needs the lock
PLAN = [
    ("pos_probe", "ewt", "pos_probe",
     "exploratory_extensions", "masked_pos_probe",
     "diffullama-ewt-posprobe-v1", False),
    ("logit_lens", "ewt", "logit_lens",
     "exploratory_extensions", "rank_logit_lens",
     "diffullama-ewt-logitlens-v1", False),
    ("attention_entropy", "ewt", "attention_entropy",
     "exploratory_extensions", "attention_entropy_over_time",
     "diffullama-ewt-entropy-v1", False),
    ("transfer_de", "de_gsd", "external_transfer",
     "external_treebank_transfer", "ewt_locked_transfer",
     "diffullama-de-transfer-v1", True),
    ("transfer_ja", "ja_gsd", "external_transfer",
     "external_treebank_transfer", "ewt_locked_transfer",
     "diffullama-ja-transfer-v1", True),
]


def run_dir_for(dataset, track, exp_id, run_id):
    return Path("results") / track / "diffullama_7b" / dataset / exp_id / run_id


results = {}
for label, dataset, experiment, track, exp_id, run_id, needs_lock in PLAN:
    run_dir = run_dir_for(dataset, track, exp_id, run_id)
    if (run_dir / "summary.json").exists():
        print(f"== {label}: already complete, skipping")
        results[label] = "skipped"
        continue

    command = [
        "dlmrel", "run",
        "--model", MODEL,
        "--dataset", f"configs/datasets/{dataset}.yaml",
        "--experiment", f"configs/experiments/{experiment}.yaml",
        "--run-id", run_id,
    ]
    if needs_lock:
        command += ["--selection-lock", str(LOCK)]
    if run_dir.exists():
        command.append("--resume")

    print("\n" + "=" * 70)
    print(f"== {label}")
    print("== " + " ".join(command))
    print("=" * 70, flush=True)

    start = time.time()
    outcome = subprocess.run(command)
    elapsed = time.time() - start
    if outcome.returncode == 0:
        results[label] = f"ok ({elapsed:.0f}s)"
    else:
        results[label] = f"FAILED (exit {outcome.returncode})"
    print(f"== {label}: {results[label]}", flush=True)

print("\nSummary")
for key, value in results.items():
    print(f"  {key:20s} {value}")


## 8. Check what landed


In [ ]:
import pandas as pd

for label, dataset, experiment, track, exp_id, run_id, _ in PLAN:
    run_dir = run_dir_for(dataset, track, exp_id, run_id)
    summary = run_dir / "summary.json"
    if not summary.exists():
        print(f"{label:20s} MISSING")
        continue
    data = json.loads(summary.read_text())
    print(
        f"{label:20s} rows={data.get('n_rows')} "
        f"sentences={data.get('n_sentences')} "
        f"status={data.get('completion_status')}"
    )

print("\nExcluded sentences per run:")
for label, dataset, experiment, track, exp_id, run_id, _ in PLAN:
    path = run_dir_for(dataset, track, exp_id, run_id) / "exclusions.parquet"
    if path.exists():
        print(f"  {label:20s} {len(pd.read_parquet(path))} excluded")

print(
    "\nWatch transfer_ja. Japanese has no whitespace word boundaries, so SentencePiece\n"
    "alignment can drop a large share of sentences. A transfer score computed over a\n"
    "small surviving subset is not comparable to the German one."
)


## 9. Download

Save this before the session ends — Colab wipes the disk on disconnect.


In [ ]:
!zip -qr /content/diffullama_runs.zip results smoke_diffullama.json

from google.colab import files

files.download("/content/diffullama_runs.zip")
